# qcc-preprocess: QuantumCircuit Input Demo

This notebook matches the README Python API section and demonstrates circuit optimization from a `QuantumCircuit` input.

In [1]:
from qiskit import QuantumCircuit
from circuit_preprocess import (
    optimize_circuit,
    optimize_circuit_with_report,
    optimize_circuit_auto_select,
    available_optimization_methods,
)

import os

_HOME = os.path.expanduser("~")
_CWD = os.getcwd()


def _sanitize_message(message: object) -> str:
    text = str(message)
    text = text.replace(_CWD, "<workspace>")
    text = text.replace(_HOME, "<home>")
    return text


def run_without_traceback(label: str, fn):
    try:
        return fn()
    except Exception as exc:
        print(f"{label} failed: {type(exc).__name__}: {_sanitize_message(exc)}")
        return None

In [3]:
# Test circuit 1: Redundant CNOTs (exploits commutation)
qc = QuantumCircuit(5)
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)
qc.cx(3, 4)
qc.cx(1, 2)
qc.rx(0.5, 0)
qc.ry(0.3, 1)
qc.rz(0.7, 2)
qc.cx(0, 2)
qc.cx(1, 3)
qc.cx(2, 4)
qc.h(range(5))

print("=== Test Circuit ===")
print("Available methods:", available_optimization_methods())
print(f"Original: depth={qc.depth()}, 2Q gates={sum(1 for inst in qc.data if len(inst.qubits) == 2)}")

=== Test Circuit ===
Available methods: ('transpile_only', 'lc', 'partitioned_lc', 'partitioned_lc_v2')
Original: depth=10, 2Q gates=9


In [4]:
# Save original circuit for comparison
qc_original = qc.copy()

In [4]:
# Compare single method in detail
qc_opt = run_without_traceback(
    "optimize_circuit",
    lambda: optimize_circuit(
        qc_original.copy(),
        method="partitioned_lc_v2",
        processors=2,
    ),
)

if qc_opt is not None:
    orig_twoq = sum(1 for inst in qc_original.data if len(inst.qubits) == 2)
    opt_twoq = sum(1 for inst in qc_opt.data if len(inst.qubits) == 2)
    print(f"\n[partitioned_lc_v2]")
    print(f"  Depth: {qc_original.depth()} → {qc_opt.depth()} (Δ{qc_opt.depth() - qc_original.depth()})")
    print(f"  2Q gates: {orig_twoq} → {opt_twoq} (Δ{opt_twoq - orig_twoq}, {100*(1-opt_twoq/orig_twoq):.1f}% reduction)")



[partitioned_lc_v2]
  Depth: 10 → 11 (Δ1)
  2Q gates: 9 → 7 (Δ-2, 22.2% reduction)


In [5]:
# Detailed report with metrics (before is aligned to original qc)
report = run_without_traceback(
    "optimize_circuit_with_report",
    lambda: optimize_circuit_with_report(
        qc_original.copy(),
        method="partitioned_lc_v2",
        processors=2,
    ),
)

if report is not None:
    orig_twoq = sum(1 for inst in qc_original.data if len(inst.qubits) == 2)
    orig_depth = qc_original.depth()
    reduction = 100 * (1 - report.twoq_after / orig_twoq) if orig_twoq > 0 else 0
    print(f"\n[Detailed Report]")
    print(f"  2Q:     {orig_twoq} → {report.twoq_after} ({reduction:.1f}% reduction)")
    print(f"  Depth:  {orig_depth} → {report.depth_after}")
    print(f"  Equiv:  {report.equivalence}")
    print(f"  Time:   {report.runtime_sec:.3f}s")



[Detailed Report]
  2Q:     9 → 7 (22.2% reduction)
  Depth:  10 → 11
  Equiv:  exact up to global phase: True (Frobenius distance=3.40e-15)
  Time:   0.010s


In [6]:
# Compare all methods - ranked by 2Q reduction (before is original qc)
print("\n=== All Methods Comparison ===")
results = []
optimized_circuits = {}  # Store all optimized results
orig_twoq = sum(1 for inst in qc_original.data if len(inst.qubits) == 2)
orig_depth = qc_original.depth()

for method in available_optimization_methods():
    r = run_without_traceback(
        f"method={method}",
        lambda m=method: optimize_circuit_with_report(qc_original.copy(), method=m, processors=2),
    )
    if r is not None:
        twoq_after = r.twoq_after
        depth_after = r.depth_after
        reduction = 100 * (1 - twoq_after / orig_twoq) if orig_twoq > 0 else 0
        results.append((method, orig_twoq, twoq_after, orig_depth, depth_after, reduction, r.runtime_sec))
        optimized_circuits[method] = r.qc_out

# Sort by 2Q reduction (descending)
results.sort(key=lambda x: x[5], reverse=True)
print("\nSelect a method to apply:")
for i, (method, twoq_b, twoq_a, d_b, d_a, red, rt) in enumerate(results):
    print(f"  [{i}] {method:16s} 2Q: {twoq_b:2d}→{twoq_a:2d} ({red:5.1f}%) depth: {d_b:2d}→{d_a:2d}  time: {rt:.3f}s")



=== All Methods Comparison ===

Select a method to apply:
  [0] transpile_only   2Q:  9→ 7 ( 22.2%) depth: 10→ 7  time: 0.007s
  [1] partitioned_lc_v2 2Q:  9→ 7 ( 22.2%) depth: 10→11  time: 0.008s
  [2] lc               2Q:  9→ 8 ( 11.1%) depth: 10→13  time: 0.007s
  [3] partitioned_lc   2Q:  9→ 9 (  0.0%) depth: 10→20  time: 0.010s


In [7]:
# Apply selected method (default: best by 2Q reduction)
if results:
    selected_index = 0  # Change this to select different method (0-based index)
    selected_method = results[selected_index][0]
    selected_reduction = results[selected_index][5]
    
    print(f"\n=== Applying Selected Method ===")
    print(f"Selected method: {selected_method} ({selected_reduction:.1f}% reduction)")
    qc = optimized_circuits[selected_method]
    print(f"qc updated with optimized circuit from: {selected_method}")



=== Applying Selected Method ===
Selected method: transpile_only (22.2% reduction)
qc updated with optimized circuit from: transpile_only
